[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/longas14/Fisica_Computacional_1/blob/main/notebooks/interpolacion_ajuste.ipynb)

---
# Física Computacional 1

**Robinson Longas Bedoya**

*Facultad de Ciencias Exactas y Aplicadas - Institución Universitaria ITM*  
[robinsonlongas@itm.edu.co](mailto:robinsonlongas@itm.edu.co) | [robinsonlongas3586@correo.itm.edu.co](mailto:robinsonlongas3586@correo.itm.edu.co)

---

### Temas principales:
* **Interpolación**
* **Ajuste de curvas**

---

### Objetivos 
1. Diferenciar conceptual y operativamente entre **interpolación** (reconstrucción exacta sobre puntos) y **ajuste de curvas** (optimización de los parámetros físicos).
2. Aplicar las rutinas de Python para la interpolación y el ajuste de datos sobre un ejemplo en particular: curvas de rotación galáctica - NGC 7331.

---

#### 📚 Bibliografía recomendada
* *Computational Physics*, Mark Newman.
* *Programming for computations - Python*, Svein Linge and Petter Langtangen.
* *Computational Physics with Python*, Eric Ayars.
* *Computational mathematics: An introduction to Numerical Analysis and Scientific Computing with Python*, Dimitrios Mitsotakis.
* *Computational physics: problem solving with Python*, Rubin H. Landau, Manuel J. Páez and Cristian C. Bordeianu.

# <span style="color:red; font-style:italic;">Interpolación</span>


En física experimental, normalmente nos encontramos con un conjunto finito y discreto de $N$ mediciones:

$$\{(x_0, y_0), (x_1, y_1), \dots, (x_{N-1}, y_{N-1})\}$$

donde $x_0 < x_1 < \dots < x_{N-1}$ corresponden a la variable independiente (tiempo, radio, frecuencia, posición) y $y_i$ al valor medido o calculado (velocidad, campo, potencial).

**Interpolar** consiste en construir una función continua $f(x)$ tal que reproduzca **con exactitud** el valor observado en cada uno de los puntos conocidos:

$$f(x_i) = y_i, \quad \forall \; i = 0, 1, \dots, N-1$$

El objetivo es estimar el comportamiento de la variable física en cualquier punto intermedio $x \in [x_0, x_{N-1}]$ donde no se tomaron mediciones.

> **Diferencia clave con la extrapolación:**
> - La **Interpolación:** Estima valores dentro del dominio de datos medidos ($x_0 \le x \le x_{N-1}$). La precisión está rigurosamente acotada por la separación de los puntos y las derivadas de la función.
> - La **Extrapolación:** Evalá fuera del dominio ($x < x_0$ o $x > x_{N-1}$). Es numéricamente inestable y carece de sustento matemático sin un modelo físico explícito.

---



### <span style="color:blue; font-style:italic;">Ejemplo: curvas de rotación de galaxias</span>

<p align="center">
  <img src="https://raw.githubusercontent.com/longas14/Fisica_Computacional_1/main/figuras/NGC7331.jpg" style="width: 65%;" alt="galaxia NGC 7331">
  <br>
</p>


Una de las primeras evidencias de la existencia de la materia oscura fueron las curvas planas de rotación de las galaxias espirales. Si asumimos que el balance total de masa de una galaxia está compuesto exclusivamente por materia luminosa, la velocidad orbital circular de las estrellas alrededor del plano galáctico debería decaer de acuerdo con un potencial kepleriano $v \sim \sqrt{ \dfrac{1}{r}}$. Sin embargo, este no es el caso: la velocidad circular apenas disminuye a radios mayores, indicando así la presencia de una nueva componente de materia no visible (materia oscura). 

En este ejemplo tomaremos un conjunto de datos discretos de la velocidad circular en función del radio para la galaxia NGC 7331 y realizaremos una interpolación por medio de las rutinas de Python.

---

In [ ]:
# carga de paquetes y diseño de estilo

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy as sp


# Seleccionar un estilo base más moderno. Yo usualmente usaba classic
plt.style.use('seaborn-v0_8-ticks') 
#plt.style.use('classic') 

# Configuración de LaTeX
plt.rcParams.update({"text.usetex": True, "text.latex.preamble": r"\usepackage{bm} \usepackage{amsmath}", "font.family": "serif","font.serif": ["Computer Modern Roman"]})

#Configuración de Ticks
plt.rcParams['xtick.major.size'] = 8
plt.rcParams['xtick.minor.size'] = 4
plt.rcParams['ytick.major.size'] = 8
plt.rcParams['ytick.minor.size'] = 4

# Forzar que las marcas apunten hacia ADENTRO (estilo paper)
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['ytick.direction'] = 'in'
plt.rcParams['xtick.top'] = True
plt.rcParams['ytick.right'] = True

In [ ]:
# carga y plot de datos experimentales

datos = np.loadtxt("../data/dataDM.txt")
r_obs = datos[:,0]
v_obs = datos[:,1]

plt.figure(figsize=(7, 5))
plt.plot(r_obs,v_obs,'r.', ms = 4)
plt.xlabel(r'$\boldsymbol{r\,\rm[kpc]}$', size=20)
plt.ylabel(r'$\boldsymbol{v \,\rm[km/s]}$', size=20)
plt.title(r'$\text{Curva de Rotación de la galaxia NGC 7331}$', fontsize=16)
plt.ylim(0, 300)
plt.yticks([0, 50, 100, 150, 200, 250, 300], [r'$0$', r'$50$', r'$100$', r'$150$', r'$200$', r'$250$', r'$300$'], fontsize=16)
plt.xticks([0, 5, 10, 15, 20, 25, 30, 35],  [r'$0$', r'$5$', r'$10$', r'$15$', r'$20$', r'$25$', r'$30$', r'$35$'],   fontsize=16)
plt.grid(True, linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()

### Discrepancia entre la curva esperada teóricamente y la curva medida experimentalmente

Si la masa estuviese confinada únicamente a la componente luminosa visible (disco y bulbo estelar), a radios mayores que el tamaño óptico del disco ($R_{\mathrm{disco}} \approx 5\text{--}6\text{ kpc}$), toda la masa quedaría encerrada: $M(r) \approx M_{\mathrm{lum}} \approx \text{cte}$.

Desde física básica se sabe que:
$$\frac{v_c^2(r)}{r} = \frac{G M_{\mathrm{lum}}}{r^2} \implies v_{\mathrm{esperada}}(r) =  \sqrt{\dfrac{GM_{\mathrm{lum}}}{r}}$$

Note que en el borde del disco se tiene que $v_{\mathrm{disco}} = \sqrt{\dfrac{G M_{\mathrm{lum}}}{r_{\mathrm{disco}}}}$. Si dividimos la velocidad en cualquier radio exterior entre la velocidad del disco, obtenemos

$$v(r) = v_{\mathrm{disco}} \sqrt{\frac{r_{\mathrm{disco}}}{r}} \quad (\text{para } r \ge r_{\mathrm{disco}})$$

A continuación, contrastamos la predicción teórica contra los datos experimentales medidos.

In [ ]:
# Definición del punto de transición del disco óptico/luminoso
# Aproximadamente donde termina la concentración del disco visible (~5.5 kpc)
r_disco = 5.5
# Velocidad media observada en la cercanía del radio del disco
v_disco = np.interp(r_disco, r_obs, v_obs) # note el uso de np.interp

# Malla para la curva teórica
r_teor = np.linspace(r_obs.min(), r_obs.max(), 500)

# Cálculo de la curva esperada:
# Dentro del disco crece aproximadamente como  (~r), fuera decae como 1/sqrt(r)
v_kepler = np.where( 
    r_teor <= r_disco, 
    v_disco * (r_teor / r_disco),           # si se cumple r_teor<= r disco, la velocidad es v~r
    v_disco * np.sqrt(r_disco / r_teor)     # si nop se cumple, v ~ 1/sqrt(r)
)

# plots 
plt.figure(figsize=(7, 5))

# curva esperada 
plt.plot(r_teor, v_kepler, '--', color='#1f77b4', lw=3,  label=r'$\text{velocidad esperada :} \; v \propto r^{-1/2}$')

# datos experimentales 
plt.plot(r_obs, v_obs, 'r.', markersize=7, label=r'$\text{velocidad observada }$')

# Resaltar la región de discrepancia (Materia Oscura)
plt.fill_between(
    r_teor[r_teor >= r_disco],
    v_kepler[r_teor >= r_disco],
    240,  # Nivel plano aproximado observado
    color='gray', alpha=0.2, label='Discrepaciancia'
)

# Línea vertical de referencia para el radio característico del disco
plt.axvline(r_disco, color='gray', linestyle=':', alpha=0.7)
plt.text(r_disco + 0.3, 50, r'Límite del disco óptico' '\n' r'$R \sim 5.5$ kpc', fontsize=11, color="#0A0909")

plt.xlabel(r'$\boldsymbol{r\,\rm[kpc]}$', size=20)
plt.ylabel(r'$\boldsymbol{v \,\rm[km/s]}$', size=20)
plt.title(r'$\text{Curva de Rotación de NGC 7331}$', fontsize=16)
plt.ylim(0, 300)
plt.xlim(0, 35)
plt.yticks([0, 50, 100, 150, 200, 250, 300], [r'$0$', r'$50$', r'$100$', r'$150$', r'$200$', r'$250$', r'$300$'], fontsize=16)
plt.xticks([0, 5, 10, 15, 20, 25, 30, 35],  [r'$0$', r'$5$', r'$10$', r'$15$', r'$20$', r'$25$', r'$30$', r'$35$'],   fontsize=16)

plt.grid(True, linestyle=":", alpha=0.6)
plt.legend(loc="lower right", framealpha=0.95, fontsize=13)
plt.tight_layout()
#plt.savefig("../figuras/curva_rotacion_ngc7331.pdf", bbox_inches="tight")
plt.show()

Note que para trazar la curva teórica necesitamos conocer la velocidad de rotación justo en el borde del disco visible ($r_{\mathrm{disco}} \sim 5.5\text{ kpc}$):

$$v_{\mathrm{kepler}}(r) = v_{\mathrm{disco}} \sqrt{\frac{r_{\mathrm{disco}}}{r}}$$

Sin embargo, en el conjunto de datos observacionales no existe un punto medido exactamente en $r = 5.5\text{ kpc}$. 

La función `np.interp(r_disco, r_obs, v_obs)` realiza una **interpolación lineal directa**:
1. Ubica automáticamente los dos radios observados más cercanos que contienen a $r_{\mathrm{disco}}$ ($r_i \le 5.5 \le r_{i+1}$).
2. Traza una recta entre ambos puntos para estimar el valor continuo $v_{\mathrm{disco}}$.

De esta manera, la curva esperada queda conectada suavemente a las observaciones en ese radio sin introducir saltos artificiales en la gráfica.

In [ ]:
# Construcción de las intepolaciones 
interp_nearest   = sp.interpolate.interp1d(r_obs, v_obs, kind="nearest")
interp_linear    = sp.interpolate.interp1d(r_obs, v_obs, kind="linear")
interp_quadratic = sp.interpolate.interp1d(r_obs, v_obs, kind="quadratic")
interp_cubic     = sp.interpolate.interp1d(r_obs, v_obs, kind="cubic")

# Malla para evaluación continua dentro del rango 
r_malla = np.linspace(r_obs.min(), r_obs.max(), 500)

v_nearest   = interp_nearest(r_malla)
v_linear    = interp_linear(r_malla)
v_quadratic = interp_quadratic(r_malla)
v_cubic     = interp_cubic(r_malla)

In [ ]:
# Plots de la interpolación
plt.figure(figsize=(7, 5))


plt.scatter(r_obs, v_obs, color="black", zorder=5, s=1, label="datos experimentales")
plt.plot(r_malla, v_nearest,   ":",  label="nearest", color="red", alpha=0.8)
plt.plot(r_malla, v_linear,    "--", label="linear ",     color="cyan", lw=5.8)
plt.plot(r_malla, v_quadratic, "-", label="quadratic ",  color="#34a02c", lw=3.8)
plt.plot(r_malla, v_cubic,     "-",  label="cubic ",      color="magenta", lw=2.2)

plt.plot(r_obs,v_obs,'r.', ms = 4)
plt.xlabel(r'$\boldsymbol{r\,\rm[kpc]}$', size=20)
plt.ylabel(r'$\boldsymbol{v \,\rm[km/s]}$', size=20)
plt.title(r'$\text{Curva de Rotación de la galaxia NGC 7331}$', fontsize=16)

plt.ylim(0, 300)
plt.yticks([0, 50, 100, 150, 200, 250, 300], [r'$0$', r'$50$', r'$100$', r'$150$', r'$200$', r'$250$', r'$300$'], fontsize=16)
plt.xticks([0, 5, 10, 15, 20, 25, 30, 35],  [r'$0$', r'$5$', r'$10$', r'$15$', r'$20$', r'$25$', r'$30$', r'$35$'],   fontsize=16)
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend(loc="lower right", framealpha=0.95, fontsize=13)
plt.tight_layout()
#plt.savefig("../figuras/curva_rotacion_ngc7331.pdf", bbox_inches="tight")
plt.show()


### Resumen:`scipy.interpolate`
> Documentación: [`scipy.interpolate`](https://docs.scipy.org/doc/scipy/reference/interpolate.html)

La biblioteca `scipy.interpolate` ofrece clases y funciones especializadas según el grado de suavidad requerido y la geometría del problema.

#### 1. Rutina clásica 1D: `scipy.interpolate.interp1d`
Construye una función interpoladora univariada evaluable como cualquier función estándar de Python. Su argumento fundamental es `kind`:

* **`kind='nearest'` (Vecino más cercano):**
  Asigna a $x$ el valor $y_i$ del nodo más próximo. Genera una función constante a trozos (escalonada).
  - *Propiedades:* Discontinua ($C^{-1}$), derivada nula casi en todo punto e indefinida en los saltos.

* **`kind='linear'` (Interpolación lineal):**
  Une nodos contiguos mediante segmentos de recta. En cada subintervalo $[x_i, x_{i+1}]$:
  $$f(x) = y_i + \frac{y_{i+1} - y_i}{x_{i+1} - x_i}(x - x_i)$$
  - *Propiedades:* Continua ($C^0$), pero su primera derivada $f'(x)$ presenta discontinuidades de salto en cada nodo.

* **`kind='quadratic'` (Spline cuadrático):**
  Ajusta polinomios de grado 2 en cada intervalo garantizando continuidad de la función y de la primera derivada ($C^1$).

* **`kind='cubic'` (Spline cúbico estándar):**
  Ajusta polinomios de grado 3 por tramos garantizando continuidad hasta la segunda derivada ($C^2$).

---

#### 2. Splines especializados de alto rendimiento: `CubicSpline` 
Es la clase más recomendada para problemas en Física es `scipy.interpolate.CubicSpline`.  Pues, esta permite:
- Controlar las **condiciones de frontera**:
  - `bc_type='natural'`: $S''(x_0) = S''(x_{N-1}) = 0$ (curvatura nula en extremos).
  - `bc_type='clamped'` (o sujeta): derivada primera impuesta $S'(x_0) = f'_0$, $S'(x_{N-1}) = f'_{N-1}$.
  - `bc_type='periodic'`: $S(x_0) = S(x_{N-1})$, $S'(x_0) = S'(x_{N-1})$, $S''(x_0) = S''(x_{N-1})$ para curvas cerradas.
- Obtener derivadas e integrales analíticas exactas de los polinomios cúbicos sin recurrir a diferencias finitas: `cs.derivative()`, `cs.integrate()`.

---

## Cálculo de la masa encerrada $M(r)$

Recordemos que bajo la aproximación de simetría esférica, se tiene que:

$$\frac{v_c^2(r)}{r} = \frac{G M(r)}{r^2} \implies M(r) = \frac{r \, v_c^2(r)}{G}$$

Podemos utilizar la función interpolada para calcular la masa $M(r)$. Antes de hacerlo, veamos una discusión importante sobre las unidades. 
### Unidades Astrofísicas
Para evitar errores de redondeo con el valor de $G$ en el SI ($6.674 \times 10^{-11} \text{ m}^3\text{kg}^{-1}\text{s}^{-2}$), se utiliza la constante gravitacional en unidades astronómicas:
- $1 \text{ kpc} \approx 3.086 \times 10^{16} \text{ km}$
- $1 M_\odot \approx 1.989 \times 10^{30} \text{ kg}$
- En estas unidades:
$$G \approx 4.3009 \times 10^{-6} \text{ kpc} \cdot (\text{km/s})^2 \cdot M_\odot^{-1}$$

Por lo tanto:
$$M(r) \approx \frac{r \, [\text{kpc}] \cdot v_c^2(r) \, [(\text{km/s})^2]}{4.3009 \times 10^{-6}} \, [M_\odot]$$

In [ ]:
# ==============================================================================
# Cálculo de la masa encerrada M(r) a partir de la interpolación 
# ==============================================================================

# Constante de gravitación en unidades galácticas: kpc, km/s, M_sun
G_astro = 4.301e-6  

# Usamos la velocidad  predicha por el Spline Cúbico v_cubic
# M(r) = (r * v^2) / G
M_encerrada = (r_malla * (v_cubic**2)) / G_astro

# Masa total encerrada al radio máximo observado
r_max = r_malla.max()
M_total = M_encerrada[-1]

print("=" * 50)
print(f"Masa total encerrada a r = {r_max:.2f} kpc:")
print(f"M_total = {M_total:.3e} Masas Solares (M_sun)")
print("=" * 50)

# plot
plt.figure(figsize=(7, 5))
plt.plot(r_malla, M_encerrada / 1e11, color="#8c4b89", lw=3.2)
plt.xlabel(r'$\boldsymbol{r\,\rm[kpc]}$', size=20)
plt.ylabel(r'$\boldsymbol{M(r) \times 10^{11}\ M_\odot}$', size=20)
plt.title("$M(r)$ en NGC 7331", fontsize=16)

plt.xlim(0, 35)
plt.ylim(0, 4.5)
plt.xticks([0, 5, 10, 15, 20, 25, 30, 35], [r'$0$', r'$5$', r'$10$', r'$15$', r'$20$', r'$25$', r'$30$', r'$35$'], fontsize=16)
plt.yticks([0, 1, 2, 3, 4], [r'$0$', r'$1$', r'$2$', r'$3$', r'$4$'], fontsize=16)

plt.grid(True, linestyle=":", alpha=0.6)
plt.tight_layout()
#plt.savefig("../figuras/curva_rotacion_ngc7331.pdf", bbox_inches="tight")
plt.show()


In [ ]:
# ==============================================================================
# Comparación de la maasa encerrada: datos interpolados vs. curva esperada (Kepler)
# ==============================================================================

# Masa encerrada observada deducida de la interpolación cúbica
M_real = (r_malla * (v_cubic**2)) / G_astro

# Masa esperada considerando v kepler
M_kepler = (r_malla * (v_kepler**2)) / G_astro

# Plot
plt.figure(figsize=(7, 5))

plt.plot(r_malla, M_real / 1e11, color="#8c4b89", lw=2.8,  label=r"$\text{Masa observada}$")
plt.plot(r_malla, M_kepler / 1e11, "--", color="#d62728", lw=2.2, label=r"$\text{Masa esperada}$")

#masa invisible necesaria
plt.fill_between(r_malla, M_kepler / 1e11, M_real / 1e11, color="gray", alpha=0.25,  label=r"$\text{Materia Oscura}$")

# radio de corte óptico
plt.axvline(r_disco, color="gray", linestyle=":", lw=1.5, alpha=0.8)
plt.text(r_disco + 0.6, 0.4, r'$\text{Disco óptico}$' '\n' r'$r \sim 5.5\ \rm{kpc}$', fontsize=12, color="#333333")


plt.xlabel(r'$\boldsymbol{r\,\rm[kpc]}$', size=20)
plt.ylabel(r'$\boldsymbol{M(r) \times 10^{11}\ [M_\odot]}$', size=20)
plt.title(r'$\text{Evidencia de Materia Oscura}$', fontsize=16)

plt.xlim(0, 35)
plt.ylim(0, 4.8)
plt.xticks([0, 5, 10, 15, 20, 25, 30, 35], [r'$0$', r'$5$', r'$10$', r'$15$', r'$20$', r'$25$', r'$30$', r'$35$'], fontsize=16)
plt.yticks([0, 1, 2, 3, 4], [r'$0$', r'$1$', r'$2$', r'$3$', r'$4$'], fontsize=16)

plt.grid(True, linestyle=":", alpha=0.6)
plt.legend(loc="upper left", framealpha=0.95, fontsize=11)
plt.tight_layout()
#plt.savefig("../figuras/masa_esperada_vs_medida_ngc7331.pdf", bbox_inches="tight")
plt.show()

---

### Materia Visible vs. Halo Oscuro

La masa total de la galaxia puede descomponerse en dos términos dominantes:

$$M_{\text{tot}}(r) = M_{\text{bar}}(r) + M_{\text{halo}}(r)$$

#### A. Materia Visible / Bariónica: $M_{\text{bar}}(r) = M_{\text{bulbo}}(r) + M_{\text{disco}}(r)$
* **Zona Interna ($r \le r_{\text{disco}}$):** La emisión de luz estelar domina la dinámica del sistema. La masa visible crece con el radio siguiendo la distribución observada:
  $$M_{\text{bar}}(r) \approx M_{\text{obs}}(r) = \frac{r \, v_{\text{obs}}^2(r)}{G}$$
* **Zona Externa ($r > r_{\text{disco}}$):** El brillo superficial del disco cae exponencialmente ($I(r) \propto e^{-r/h_R}$) y la densidad del gas disminuye drásticamente. Más allá del borde visible ($r_{\text{disco}} \approx 5.5\text{ kpc}$), ya no se agrega masa luminosa significativa:
  $$M_{\text{bar}}(r) = M_{\text{bar, max}} \approx \text{constante}$$
* **Predicción Kepleriana:** Si no existiera materia adicional en la periferia, para $r > r_{\text{disco}}$ el potencial se comportaría como el de una masa puntual ($M \approx \text{cte}$), forzando a las velocidades a decaer keplerianamente:
  $$v(r) \propto \frac{1}{\sqrt{r}}$$

#### B. Materia Oscura (Halo): $M_{\text{halo}}(r)$
Como las curvas observadas no caen sino que permanecen planas ($v_{\text{obs}} \approx v_0 \approx \text{cte}$), la masa observada continúa creciendo linealmente con la distancia:

$$M_{\text{obs}}(r) \propto r \cdot v_0^2 \propto r$$

El déficit entre la masa total requerida y la masa visible corresponde al halo de materia oscura:

$$M_{\text{halo}}(r) = M_{\text{obs}}(r) - M_{\text{bar}}(r)$$

En la periferia de galaxias espirales gigantes como NGC 7331, la masa del halo termina aportando entre el $75\%$ y el $85\%$ de la masa total encerrada.

In [ ]:
# =======================================================================
# Descomposición de la masa: materia visible (bulbo + disco) vs. Halo
# =======================================================================

# Radio de corte de la materia visible (límite del disco óptico)
r_disco = 5.5   # kpc
idx_corte = np.argmin(np.abs(r_malla - r_disco)) # indice de la posición para el valor mínimo

# Masa bariónica (Bulbo + Disco)
# Crece con la curva observada hasta r_disco y se congela en valor constante
M_bar = np.copy(M_real) # crear copia de M_real
M_bar_total = M_real[idx_corte] # la masa bulbo + disk va hasta el corte. Acá toma el valor de la masa en ese corte
M_bar[r_malla > r_disco] = M_bar_total # si r<rdisk  la masa crece. Si r>rdisk, se congela la masa en el corte.

# Masa total del halo de materia oscura
M_halo = np.maximum(M_real - M_bar, 0.0)

# Masas evaluadas en el borde de la galaxia (r_max)
idx_max = -1 #ultimo elemento del arreglo
r_max = r_malla[-1]

M_tot_obs = M_real[idx_max]
M_bar_max = M_bar[idx_max]
M_halo_max = M_halo[idx_max]

fraccion_halo = (M_halo_max / M_tot_obs) * 100.0
fraccion_bar  = (M_bar_max / M_tot_obs) * 100.0

print("=" * 65)
print("BALANCE DE MASA: MATERIA VISIBLE (BULBO + DISCO) vs. HALO")
print("=" * 65)
print(f"Radio del disco óptico (r_disco) = {r_disco:.1f} kpc")
print(f"Radio exterior evaluado (r_max)  = {r_max:.1f} kpc")
print("-" * 65)
print(f"Masa Total                         = {M_tot_obs / 1e11:.2f} x 10^11 M_sun")
print(f"Masa visible total (Bulbo + Disco) = {M_bar_max / 1e11:.2f} x 10^11 M_sun  ({fraccion_bar:.1f} %)")
print(f"Masa del Halo de Materia Oscura    = {M_halo_max / 1e11:.2f} x 10^11 M_sun  ({fraccion_halo:.1f} %)")
print("=" * 65)

# plot
plt.figure(figsize=(7, 5))

# Masa total deducida de los datos observados
plt.plot(r_malla, M_real / 1e11,  color="#8c4b89",  lw=2.8, label=r"$\text{Masa total observada } M_{\mathrm{obs}}(r)$")

# Componente de materia visible (Bulbo + Disco)
plt.plot(
    r_malla, M_bar / 1e11, "--", color="#d62728", lw=2.2, label=r"$\text{Materia visible } (M_{\mathrm{bulbo}} + M_{\mathrm{disco}})$")

# Componente del halo de materia oscura
plt.plot( r_malla, M_halo / 1e11, "-.",  color="#2ca02c",   lw=2.2, label=r"$\text{Halo de Materia Oscura } M_{\mathrm{halo}}(r)$")

# Sombreado de la brecha dominada por materia oscura
plt.fill_between(r_malla, M_bar / 1e11, M_real / 1e11, color="#2ca02c",  alpha=0.18)

# Línea vertical del radio de corte óptico
plt.axvline(r_disco, color="gray", linestyle=":", lw=1.5, alpha=0.8)
plt.text( r_disco + 0.6, 0.45, r"$\text{Límite óptico}$" "\n" r"$r \sim 5.5\ \mathrm{kpc}$",fontsize=11, color="#333333")

# etiquetas
plt.xlabel(r"$\boldsymbol{r\,\mathrm{[kpc]}}$", size=18)
plt.ylabel(r"$\boldsymbol{M(r) \times 10^{11}\ [M_\odot]}$", size=18)
plt.title(r"$\text{Descomposición de Masa Dinámica en NGC 7331}$", fontsize=14, pad=12)

plt.xlim(0, 35)
plt.ylim(0, 4.8)
plt.xticks([0, 5, 10, 15, 20, 25, 30, 35], [r"$0$",r"$5$",r"$10$",r"$15$",r"$20$",r"$25$",r"$30$",r"$35$"],fontsize=14)
plt.yticks([0, 1, 2, 3, 4],[r"$0$", r"$1$", r"$2$", r"$3$", r"$4$"], fontsize=14)

plt.grid(True, linestyle=":", alpha=0.6)
plt.legend(loc="upper left", framealpha=0.95, fontsize=10)
plt.tight_layout()

#plt.savefig("../figuras/descomposicion_masa_ngc7331.pdf", bbox_inches="tight")
plt.show()

---
# <span style="color:red; font-style:italic;">Ajuste de curvas</span>

Si bien la **interpolación** nos permite reconstruir una función continua a partir de datos experimentales y calcular derivadas e integrales exactas (la 
mejor opción es usando `CubicSpline`), **no constituye una prueba de hipótesis física**: simplemente obliga a una curva a pasar por puntos experimentales que contienen incertidumbre observacional e instrumental. 

Para evaluar qué tan "bueno" es un modelo teórico frente a los datos experimentales, se recurre al **ajuste de curvas** (*fitting*). En física experimental y computacional existen dos metodologías estándar:
1. **Máxima Verosimilitud (*Maximum Likelihood Estimation*, MLE):** ampliada comúnmente en análisis bayesianos y cadenas de Markov Monte Carlo (MCMC).
2. **Mínimos Cuadrados (*Least Squares* / $\chi^2$):** que minimiza la suma ponderada del cuadrado de los residuos.

Ambos métodos (o al menos el de mínimos cuadrados) se estudiarán en detalle en el siguiente curso. En esta práctica ilustraremos cómo opera un ajuste no lineal mediante mínimos cuadrados utilizando `scipy.optimize.curve_fit` sobre modelos de distribución de materia oscura.

---

### Familias de Perfiles de Materia Oscura: *Cusps* vs. *Cores*

La distribución espacial de la materia oscura suele modelarse a través de perfiles radiales de densidad $\rho(r)$. En la literatura astrofísica existen dos grandes vertientes según su comportamiento hacia el centro galáctico ($r \to 0$):

#### 1. Perfiles con Cúspide (*Cuspy Profiles*) — Predicciones de simulaciones cosmológicas $\Lambda\text{CDM}$
En estos modelos, la densidad central diverge formalmente hacia el infinito ($\lim_{r \to 0} \rho(r) \to \infty$), aunque la masa total encerrada $M(r)$ permanece finita:
* **Navarro-Frenk-White (NFW):** Estándar en simulaciones de $N$-cuerpos de materia oscura fría con decaimiento exterior $\rho \propto r^{-3}$.  
  > Navarro, J. F., Frenk, C. S., & White, S. D. (1996). [*The Structure of Cold Dark Matter Halos*](https://arxiv.org/abs/astro-ph/9508025). *The Astrophysical Journal*, 462, 563.
* **Perfil de Einasto:** Modelo donde la pendiente logarítmica de la densidad varía continuamente como una ley de potencias en el radio.  
  > Einasto, J. (1965). [*On the Construction of a Composite Model for the Galaxy and on the Determination of the System of Galactic Parameters*](https://ui.adsabs.harvard.edu/abs/1965TrAlm...5...87E/abstract). *Trudy Astrofizicheskogo Instituta Alma-Ata*, 5, 87.  
* **Perfil de Hernquist:** Solución analítica con cúspide central ($\rho \propto r^{-1}$) y caída exterior rápida ($\rho \propto r^{-4}$), ideal para modelos dinámicos exactos de bulbos y elípticas.  
  > *Ref.:*Hernquist, L. (1990). [*An analytical model for spherical galaxies and bulges*](https://ui.adsabs.harvard.edu/abs/1990ApJ...356..359H/abstract). *The Astrophysical Journal*, 356, 359.
* **Review para halos DM** Navarro et al. 2004, [*arXiv:astro-ph/0311231*](https://arxiv.org/abs/astro-ph/0311231)


#### 2. Perfiles con Núcleo Plano (*Cored Profiles*) — Ajustes fenomenológicos y observacionales
En estos modelos, la densidad central satura en un valor finito y constante ($\lim_{r \to 0} \rho(r) = \rho_0$), prediciendo $v \propto r$ en el centro:
* **Esfera Pseudo-Isotérmica (PIS):** Solución simple con velocidad asintótica constante a grandes distancias ($\rho \propto r^{-2}$).  
  > Begeman, K. G., Broeils, A. H., & Sanders, R. H. (1991). [*Extended rotation curves of spiral galaxies - Dark haloes and modified dynamics*](https://ui.adsabs.harvard.edu/abs/1991MNRAS.249..523B/abstract). *Monthly Notices of the Royal Astronomical Society*, 249(3), 523–537.
* **Perfil de Burkert:** Perfil con núcleo suave en el centro que replica la caída externa $\rho \propto r^{-3}$ de las simulaciones NFW.  
  > Burkert, A. (1995). [*The Structure of Dark Matter Halos in Dwarf Galaxies*](https://arxiv.org/abs/astro-ph/9504041). *The Astrophysical Journal Letters*, 447, L25.
* **Familia Generalizada $(\alpha, \beta, \gamma)$ de Zhao:** Parametrización universal de cinco parámetros que engloba a NFW, Hernquist y núcleos planos como casos particulares.  
  > Zhao, H. (1996). [*Analytical models for galactic nuclei*](https://ui.adsabs.harvard.edu/abs/1996MNRAS.278..488Z/abstract). *Monthly Notices of the Royal Astronomical Society*, 278(2), 488–496.

En esta sesión estudiaremos el modelo **Pseudo-Isotérmico** y veremos como se ajustan los parámetros libres del modelo con los datos experimentales.

---

### Modelo Físico: Halo Pseudo-Isotérmico con Núcleo Plano (*Core*)

En una galaxia espiral en equilibrio hidrodinámico y simetría esférica, el balance entre la aceleración centrípeta y la atracción gravitacional neta exige:
$$\frac{v_c^2(r)}{r} = \frac{G M(r)}{r^2} \implies v_c(r) = \sqrt{\frac{G M(r)}{r}}$$

El perfil de densidad volumétrica del halo pseudo-isotérmico está parametrizado por:
$$\rho(r) = \frac{\rho_0}{1 + \left(\dfrac{r}{r_c}\right)^2}$$

donde:
- $\rho_0$: densidad volumétrica central de materia oscura ($r \to 0$).
- $r_c$: radio característico del núcleo plano (*core radius*).

Integrando la masa encerrada en una esfera de radio $r$:
$$M = \int_V \rho \,dV \implies M(r) = 4\pi \int_0^r \rho(r') r'^2 dr' = 4\pi \rho_0 r_c^3 \left[ \frac{r}{r_c} - \arctan\left(\frac{r}{r_c}\right) \right]$$

Sustituyendo $M(r)$ en la ecuación de velocidad circular:
$$v_c^2(r) = 4\pi G \rho_0 r_c^2 \left[ 1 - \frac{r_c}{r} \arctan\left(\frac{r}{r_c}\right) \right]$$

En el régimen asintótico exterior ($r \gg r_c$), se tiene que $\arctan(r/r_c) \to \pi/2$ y el cociente $r_c/r \to 0$. De este modo, la velocidad satura en una meseta plana constante $v_0$:
$$v_0 \equiv \lim_{r \to \infty} v_c(r) = \sqrt{4\pi G \rho_0 r_c^2} \implies \rho_0 = \frac{v_0^2}{4\pi G r_c^2}$$

Reescribiendo la velocidad en términos de los observables directos, la función de ajuste depende únicamente de dos parámetros libres $(v_0, r_c)$:
$$v_c(r; v_0, r_c) = v_0 \sqrt{1 - \frac{r_c}{r} \arctan\left(\frac{r}{r_c}\right)}$$

In [ ]:
# Definición analítica del modelo físico del halo
def halo_pseudoisotermico(r, rc, v0):
    """
    Velocidad circular producida por un halo pseudo-isotérmico.
    r  : radio galactocéntrico (kpc)
    rc : radio del núcleo plano o core (kpc)
    v0 : velocidad asintótica plana (km/s)
    """
    # Evitar indeterminación por división entre cero en el origen r = 0
    r_seguro = np.where(r == 0, 1e-6, r) # si r vale cero, reemplazar po 1e-6, si no, dejelo como r
    radicando = 1.0 - (rc / r_seguro) * np.arctan(r_seguro / rc)
    return v0 * np.sqrt(np.maximum(radicando, 0.0)) # para evitar raíces negativas. 

# aislamos la componente de velocidad del Halo (v_halo)
G_astro = 4.3009e-6 # G en unidades galácticas: kpc * (km/s)^2 / M_sun

# Velocidad circular originada exclusivamente por la materia oscura
# Usamos r_fine y M_halo obtenidos en la descomposición de masas
r_seguro_malla = np.where(r_malla == 0, 1e-6, r_malla)
v_halo_malla = np.sqrt( np.maximum((G_astro * M_halo) / r_seguro_malla, 0.0) )

#  Filtro para excluir la región central dominada por el bulbo y centrarnos en el halo
r_corte = 5.5  # kpc
mascara_halo = r_malla >= r_corte

r_fith = r_malla[mascara_halo]
v_fith = v_halo_malla[mascara_halo]


#ajuste por mínimos cuadrados no lineales sobre v_halo

# Estimación inicial de parámetros (p0), v0~ 240 y r0~2
p0 = [3.5, 230.0]

# popt: parameters optimal:  son las coordenadas del punto optimo
# pcov : parameter covariance matrix: es la matriz 2x2 de covarianza
popt, pcov = sp.optimize.curve_fit(halo_pseudoisotermico, r_fith, v_fith, p0=p0) # primero la variable independiente

# Extracción de parámetros óptimos e incertidumbres a 1-sigma
rc_fith, v0_fith = popt
sigma_rc = np.sqrt(pcov[0, 0])
sigma_v0 = np.sqrt(pcov[1, 1])

print("=" * 60)
print("PARÁMETROS FÍSICOS DEL HALO (NGC 7331):")
print(f"  v_0 (velocidad asintótica) = {v0_fith:.2f} +/- {sigma_v0:.2f} km/s")
print(f"  r_c (radio del core)       = {rc_fith:.2f} +/- {sigma_rc:.2f} kpc")
print("=" * 60)

# Evaluación del modelo ajustado sobre la malla continua
v_fit = halo_pseudoisotermico(r_malla, *popt)

# plot 
plt.figure(figsize=(7, 5))

# Datos observacionales
plt.plot(r_obs, v_obs, "k.", ms=5, zorder=5, label=r"$\text{Datos experimentales}$")

# Interpolación de referencia v_cubic
plt.plot(r_malla, v_cubic, "--", color="#8c4b89", lw=2.0, alpha=0.8, label=r"$\text{Spline Cúbico }$")

# Velocidad extraída de materia oscura
plt.plot(r_malla, v_halo_malla, ":",color="#1f77b4", lw=2.0, label=r"$v_{\mathrm{halo}}\ \text{extraída de } M_{\mathrm{halo}}$")

# Modelo físico (pseudo-isotérmico) ajustado
plt.plot(r_malla, v_fit, "-", color="#2ca02c", lw=2.6,  label=rf"$\text{{Halo PIS: }} v_0 = {v0_fith:.1f}\ \rm{{km/s}},\ r_c = {rc_fith:.2f}\ \rm{{kpc}}$")

# Línea del corte del disco
plt.axvline(r_corte, color="gray", linestyle=":", lw=1.3, alpha=0.8)

# estilo 
plt.xlabel(r'$\boldsymbol{r\,\rm[kpc]}$', size=20)
plt.ylabel(r'$\boldsymbol{v \,\rm[km/s]}$', size=20)
plt.title(r'$\text{Ajuste del Halo de Materia Oscura en NGC 7331}$', fontsize=16)

plt.xticks([0, 5, 10, 15, 20, 25, 30, 35],  [r'$0$', r'$5$', r'$10$', r'$15$', r'$20$', r'$25$', r'$30$', r'$35$'], fontsize=16)
plt.yticks([0, 50, 100, 150, 200, 250, 300], [r'$0$', r'$50$', r'$100$', r'$150$', r'$200$', r'$250$', r'$300$'], fontsize=16)

plt.grid(True, linestyle=":", alpha=0.6)
plt.legend(loc="lower right", framealpha=0.95, fontsize=10.5)
plt.tight_layout()
plt.savefig("../figuras/ajuste_halo_ngc7331.pdf", bbox_inches="tight")
plt.show()

---
### 📌 Resumen: `scipy.optimize.curve_fit`
> Documentación oficial: [`scipy.optimize.curve_fit`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.curve_fit.html)

La función `curve_fit` implementa el método de **mínimos cuadrados no lineales** (por defecto mediante el algoritmo de **Levenberg-Marquardt**) para encontrar el conjunto de parámetros óptimos $\mathbf{p}^*$ que minimiza la suma ponderada del cuadrado de los residuos:

$$\chi^2(\mathbf{p}) = \sum_{i=1}^{N} \left[ \frac{y_i - f(x_i; \mathbf{p})}{\sigma_i} \right]^2$$

---

#### Sintaxis General

```python
popt, pcov = curve_fit(f, xdata, ydata, p0=None, sigma=None, absolute_sigma=False, **kwargs) 
```
---

#### Parámetros de entrada

* **`f` (función objetivo / modelo físico):** Función matemática a ajustar. Debe recibir la variable independiente como primer argumento y luego los parámetros libres a optimizar: `f(x, p1, p2, ...)`.
* **`xdata`:** Arreglo con los valores de la variable independiente (en nuestro caso, el radio galactocéntrico $r_{\text{obs}}$).
* **`ydata`:** Arreglo con los datos experimentales u observacionales observados (la velocidad circular medida $v_{\text{obs}}$).
* **`p0` (*initial guess*):** Lista o tupla con los valores iniciales de prueba para cada parámetro libre `[p1_0, p2_0, ...]`. Un punto de partida físicamente razonable evita que el algoritmo converja a mínimos locales no físicos.
* **`sigma` (opcional):** Incertidumbres experimentales asociadas a `ydata`. Si no se especifica, todos los puntos tienen el mismo peso estadístico ($w_i = 1$).
* **`absolute_sigma` (booleano):**
  * `False` (por defecto): Trata a `sigma` como ponderaciones relativas y reescala la covarianza según la dispersión residual del ajuste.
  * `True`: Asume que los valores de `sigma` son barras de error absolutas en unidades físicas reales.

---

#### Salidas de la función

1. **`popt` (*parameters optimal*):**
   * Arreglo 1D con los mejores parámetros encontrados por el optimizador.
   * Respeta exactamente el orden en que fueron declarados en la definición de `f`.
   * Para dos parámetros libres: `p1_fit, p2_fit = popt`.

2. **`pcov` (*parameter covariance matrix*):**
   * Matriz cuadrada de $M \times M$ (donde $M$ es el número de parámetros libres) que describe la varianza e interdependencia estadística del ajuste:

$$\text{pcov} = \begin{pmatrix} \sigma_{p_1}^2 & \operatorname{Cov}(p_1, p_2) \\ \operatorname{Cov}(p_2, p_1) & \sigma_{p_2}^2 \end{pmatrix}$$

   * **Diagonal principal:** Contiene las varianzas estadísticas marginales de cada parámetro ($\sigma_{p_i}^2 = \text{pcov}[i, i]$).
   * **Incertidumbre estándar ($1\sigma$):** Se calcula extrayendo la raíz cuadrada de la diagonal principal:
     ```python
     sigma_p = np.sqrt(np.diag(pcov))
     ```
     Bajo una aproximación asintóticamente gaussiana, este valor representa el intervalo de confianza clásico al **$68.3\%$**.
   * **Elementos fuera de la diagonal:** Representan las covarianzas cruzadas $\operatorname{Cov}(p_i, p_j)$, las cuales cuantifican el grado de acoplamiento o correlación entre los parámetros libres durante la optimización.

### <span style="color:blue; font-style:italic;">Aplicación en nuestro ejemplo</span> 


#### Definición de la función teórica  (`halo_pseudoisotermico`)
* **Orden de argumentos:** La primera variable **siempre** debe ser la variable independiente (en este caso, el radio $r$). Los argumentos siguientes (`rc`,`v0`) corresponden a los parámetros libres que queremos optimizar.
* **Manejo de singularidades con (`np.where`):** Para evitar indeterminaciones por división entre cero en el origen ($r = 0$), se sustituye temporalmente cualquier valor nulo por un valor de  $10^{-6}\text{ kpc}$.
* **Estabilidad del radicando (`np.maximum`):** Garantiza que el interior de la raíz cuadrada no tome valores negativos durante las iteraciones de búsqueda, evitando cálculos con números complejos o `NaN`.

#### Semilla inicial de búsqueda (`p0`)
El algoritmo de optimización no lineal requieren un punto de partida para iterar. En lugar de elegir valores arbitrarios, observamos la tendencia en la gráfica de datos:
* La transición entre la subida inicial y la zona plana ocurre en los primeros kilopársecs $\rightarrow$ elegimos un radio de núcleo de prueba $r_c = 2.0\text{ kpc}$.
* La curva de rotación se estabiliza alrededor de los $240\text{ km/s}$ a radios grandes $\rightarrow$ elegimos $v_0 = 240.0\text{ km/s}$.

#### Ejecución y extracción de resultados (`popt` y `pcov`)
La función `curve_fit` recibe el modelo, los datos $(r_{\text{obs}}, v_{\text{obs}})$ y los valores iniciales `p0`. Devuelve dos salidas:
1. **`popt` (*parameters optimal*):** Vector con los valores optimizados de los parámetros libres ($r_c$ en `popt[0]` y $v_0$ en `popt[1]`).
2. **`pcov` (*parameter covariance*):** Matriz de covarianza del ajuste. Los elementos de la diagonal principal corresponden a las varianzas estadísticas de cada parámetro ($\sigma_{r_c}^2 = \text{pcov}[0, 0]$ y $\sigma_{v_0}^2 = \text{pcov}[1, 1]$). Al aplicar la raíz cuadrada (`np.sqrt`), obtenemos las incertidumbres estándar a $1\sigma$.


### Determinación de la Densidad Central, $\rho_0$,

Finalmente, con los parámetros óptimos $(r_c,v_0)$ obtenidos del ajuste, calculamos la densidad central del halo mediante:

$$\rho_0 = \frac{v_0^2}{4\pi G r_c^2}$$

### Propagación de incertidumbres

#### caso realista
Debido a que $v_0$ y $r_c$ provienen de un ajuste no lineal acoplado, su incertidumbre no es independiente. La varianza de $\rho_0$ requiere incluir el término de covarianza cruzada $\operatorname{Cov}(r_c, v_0) = \text{pcov}[0, 1]$:

$$\sigma_{\rho_0}^2 = \left( \frac{\partial \rho_0}{\partial v_0} \right)^2 \sigma_{v_0}^2 + \left( \frac{\partial \rho_0}{\partial r_c} \right)^2 \sigma_{r_c}^2 + 2 \left( \frac{\partial \rho_0}{\partial r_c} \right)  \left( \frac{\partial \rho_0}{\partial v_0} \right) \operatorname{Cov}(r_c, v_0)$$

donde las derivadas parciales analíticas son:

$$\frac{\partial \rho_0}{\partial v_0} = \frac{2\rho_0}{v_0}, \qquad \frac{\partial \rho_0}{\partial r_c} = -\frac{2\rho_0}{r_c}$$

#### aproximación mala
Si asumimos que los errores son independiente, simplemente sumamos el error en cuadratura como lo hacemos en los laboratorios de física básica. 
Es decir, a partir de las incertidumbres $\sigma_{v_0}$ y $\sigma_{r_c}$ obtenidas del ajuste, propagamos el error relativo estándar:

$$\sigma_{\rho_0} = \rho_0 \sqrt{\left( 2 \frac{\sigma_{v_0}}{v_0} \right)^2 + \left( 2 \frac{\sigma_{r_c}}{r_c} \right)^2}$$

#### aproximación muy mala
Podemos reportar simplemente el valor central de $\rho_0$ (cosa que no es recomendada, pues toda medida tiene errores asociados) con los valores óptimos $v_0$ y $r_c$ determinados por el ajuste,
$$\rho_0 = \frac{v_0^2}{4\pi G r_c^2}$$

In [ ]:
# ==============================================================================
# Cálculo rho_0 y Propagación de Incertidumbres
# ==============================================================================

# Constante de gravitación en unidades galácticas:
# G ~ 4.301e-6 kpc * (km/s)^2 / M_sun
G_astro = 4.301e-6  

# Valor central de rho_0 usando la relación analítica:
# rho_0 = v0^2 / (4 * pi * G * rc^2)
rho0_fit = (v0_fith**2) / (4.0 * np.pi * G_astro * (rc_fith**2))

# Propagación formal de errores usando la matriz de covarianza (pcov):
# Derivadas parciales analíticas respecto a los parámetros ajustados:
#   d(rho0)/d(rc) = -2 * v0^2 / (4 * pi * G * rc^3) = -2 * rho0 / rc
#   d(rho0)/d(v0) =  2 * v0 / (4 * pi * G * rc^2) =  2 * rho0 / v0
d_rho_drc = -2.0 * rho0_fit / rc_fith
d_rho_dv0 =  2.0 * rho0_fit / v0_fith

# Varianza de rho_0 considerando varianzas individuales y covarianza cruzada:
# sigma_rho0^2 = (d_rho/dv0)^2 * Var(v0) + (d_rho/drc)^2 * Var(rc) + 2 * (d_rho/dv0) * (d_rho/drc) * Cov(v0, rc)
var_rho0 = ( (d_rho_drc**2) * pcov[0, 0] + (d_rho_dv0**2) * pcov[1, 1] + 2.0  * d_rho_drc * d_rho_dv0 * pcov[0, 1] )
sigma_rho0 = np.sqrt(var_rho0)

# ------
print("=" * 60)
print("PROPIEDADES CENTRALES DEL HALO DE MATERIA OSCURA:")
print(f"  v_0 (asintótica)     = {v0_fith:.2f} +/- {sigma_v0:.2f} km/s")
print(f"  r_c (radio de núcleo)= {rc_fith:.2f} +/- {sigma_rc:.2f} kpc")
print("-" * 60)
print(f"  rho_0 (densidad central a 1 sigma):")
print(f"    = ({rho0_fit:.3e} +/- {sigma_rho0:.3e}) M_sun / kpc^3")
print(f"    = {rho0_fit / 1e7:.2f} x 10^7 M_sun / kpc^3")

# ==============================================================================
# Conversión al Sistema Internacional: kg / m^3
# ==============================================================================
M_SUN_KG = 1.9884e30          # 1 M_sun en kg
KPC_TO_M = 3.0857e19          # 1 kpc en metros
FACTOR_CONVERSION = M_SUN_KG / (KPC_TO_M**3)  # ~ 6.768e-29 kg/m^3 por cada M_sun/kpc^3

# Conversión de rho_0 y su incertidumbre
rho0_si = rho0_fit * FACTOR_CONVERSION
sigma_rho0_si = sigma_rho0 * FACTOR_CONVERSION

print("-" * 60)
print(f"Densidad central rho_0 en el SI:")
print(f"  rho_0 = ({rho0_si:.3e} +/- {sigma_rho0_si:.3e}) kg/m^3")
print("-" * 60)

# ==============================================================================
# Conversión a unidades de Física de Partículas: GeV / cm^3
# ==============================================================================

# Constantes fundamentales
C_LUZ = 2.99792458e8              # m/s
EV_A_JOULE = 1.602176634e-19      # J/eV
GEV_A_KG = (1e9 * EV_A_JOULE) / (C_LUZ**2)  # ~ 1.782662e-27 kg por cada GeV/c^2

# Factor de conversión directo desde kg/m^3 a GeV/cm^3
FACTOR_SI_A_GEV = 1.0 / (GEV_A_KG * 1e6)    # ~ 5.60959e20 (GeV/cm^3) / (kg/m^3)

# Conversión de la densidad central
rho0_gev = rho0_si * FACTOR_SI_A_GEV
sigma_rho0_gev = sigma_rho0_si * FACTOR_SI_A_GEV

print("-" * 60)
print("DENSIDAD CENTRAL rho_0 EN FÍSICA DE PARTÍCULAS:")
print(f"  rho_0 = ({rho0_gev:.3f} +/- {sigma_rho0_gev:.3f}) GeV / cm^3")
print("=" * 60)

In [ ]:
# ==============================================================================
# Perfil radia para la sensidad del Halo de Materia Oscura 
# ==============================================================================

# Perfil analítico en GeV / cm^3
rho_perfil = rho0_gev / (1.0 + (r_malla / rc_fith) ** 2)

plt.figure(figsize=(7, 5))

# Curva de densidad
plt.plot( r_malla, rho_perfil, color="#2ca02c", lw=2.6, label=r"$\rho(r) = \frac{\rho_0}{1 + (r/r_c)^2}$")

# Punto característico en r = rc
plt.axvline(rc_fith, color="gray", linestyle=":", lw=1.3, alpha=0.8)
plt.axhline(rho0_gev / 2.0, color="gray", linestyle=":", lw=1.3, alpha=0.8)
plt.scatter([rc_fith], [rho0_gev / 2.0], color="#d62728", s=50, zorder=5,label=rf"$r_c = {rc_fith:.2f}\ \rm{{kpc}}\ (\rho_0/2)$")

# Línea del corte óptico
plt.axvline(r_corte, color="#8c4b89", linestyle="--", lw=1.3, alpha=0.7)
plt.text(r_corte + 0.5,rho0_gev * 0.85,r"$\text{Límite óptico}$" "\n" r"$r \sim 5.5\ \rm{kpc}$",fontsize=10, color="#8c4b89")


plt.xlabel(r"$\boldsymbol{r\,\rm{[kpc]}}$", size=18)
plt.ylabel( r"$\boldsymbol{\rho(r)\ \rm{[GeV/cm^3]}}$", size=18 )
plt.title(  r"$\text{Perfil de densidad del halo de materia oscura en la hiótesis pseudo-isotérmica}$", fontsize=14,  pad=10)

plt.xlim(0, 35)
plt.ylim(0, rho0_gev * 1.08)
plt.xticks([0, 5, 10, 15, 20, 25, 30, 35], fontsize=14)
plt.yticks(fontsize=14)

plt.grid(True, linestyle=":", alpha=0.6)
plt.legend(loc="upper right", framealpha=0.95, fontsize=11)
plt.tight_layout()
plt.savefig("../figuras/perfil_densidad_ngc7331.pdf", bbox_inches="tight")
plt.show()

### 🌌 Conclusión: El análisis de las curvas de rotación galácticas conduce a dos opciones 

1. **Hipótesis de la masa faltante: materia oscura** Si la gravitación estándar es válida a escalas astronómicas, existe una cantidad significativa de materia no luminosa que domina el potencial gravitacional: **la materia oscura**.

2. **Hipótesis de la gravedad modificada:** Si no existe dicha masa invisible, las leyes de la gravedad aceptadas (newtoniana o relatividad general) presentan fallas en regímenes de muy baja aceleración y requieren ser modificadas.

---

<div style="display: flex; justify-content: center; gap: 15px;">
    <img src="https://raw.githubusercontent.com/longas14/Fisica_Computacional_1/main/figuras/DMhalo.jpeg" style="width: 48%; object-fit: contain;">
     <img src="https://raw.githubusercontent.com/longas14/Fisica_Computacional_1/main/figuras/DMhalo2.jpg" style="width: 48%; object-fit: contain;">
</div>

# 📌 Problema propuesto



Cuando una curva en el plano se cierra sobre sí misma o presenta trayectorias donde a un mismo valor de $x$ le corresponden múltiples valores de $y$, la interpolación cartesiana estándar $y = f(x)$ falla por no ser una función univaluada. La solución computacional consiste en parametrizar la curva en términos de una variable auxiliar independiente $t$ (por ejemplo, el índice o la longitud de arco del recorrido), generando dos interpolaciones independientes: $x(t)$ e $y(t)$.

---

1. Descargue el archivo de datos provisto (`puntos_silueta.txt`), el cual contiene las columnas de coordenadas de los nodos ordenados:
   $$\{(x_0, y_0), (x_1, y_1), \dots, (x_N, y_N)\}$$
   donde el último punto coincide con el inicial para garantizar una trayectoria cerrada.

2. * Defina un parámetro temporal/posicional discreto $t \in [0, N]$.
   * Empleando `scipy.interpolate.CubicSpline`, construya dos funciones interpolantes independientes $x(t)$ e $y(t)$.
   * Use la condición de frontera `bc_type='periodic'` para garantizar continuidad de clase $C^2$ (continuidad en la curva, en la tangente y en la curvatura) en el punto de cierre.

3. * Genere una malla fina de al menos 1000 puntos para $t \in [0, N]$ y evalúe las funciones para obtener $(x_{\text{interp}}, y_{\text{interp}})$.
   * Elabore una figura en Matplotlib que incluya:
   * Los nodos discretos originales como puntos destacados (`scatter`).
   * La trayectoria continua interpolada como una curva suave.
   * Cuadrícula, etiquetas de ejes, leyenda y la instrucción `plt.axis('equal')` para conservar la relación de aspecto geométrica sin distorsionar la figura.

---

